# Lenguajes y Autómatas II (SCD-1016)
## Sesión 26 · TAC para Llamadas a Funciones y Retornos

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué instrucciones intermedias hacen falta para llamar a una función y recoger su resultado?
>
> **Objetivo de la sesión:** Emitir la secuencia PARAM ... CALL ... RETURN que representa una invocación, incluyendo llamadas anidadas y funciones sin valor de retorno.

**Entregable:** tus funciones emitir_llamada() y emitir_return() en artesano_sesion26_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion26.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

emitir_llamada() agrega a lineas los PARAM necesarios y la instrucción CALL, actualizando *n. emitir_return() escribe en dst el retorno con o sin valor.

Al final de la sesión tendrás: **Dos funciones en C que emiten la secuencia de una llamada y la instrucción de retorno, verificadas con seis casos.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 26](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion26.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion26.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion26.h"

int emitir_llamada(char lineas[][64], int max, int *n, const char *funcion, const char *a1, const char *a2, const char *resultado);
void emitir_return(char *dst, int cap, const char *valor);
```

- Cada argumento no NULL genera un PARAM, en el orden a1, a2.
- El número que acompaña a CALL es la cantidad real de PARAM emitidos.
- Si resultado es NULL, el CALL no asigna; si el valor de retorno es NULL, escribe solo RETURN.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion26_c.c` junto con `artesano_sesion26_c.c`
> (o `ia_sesion26_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `PARAM x` | `PARAM x` |
| 2 | `PARAM y` | `PARAM y` |
| 3 | `CALL con dos argumentos` | `_t0 = CALL f, 2` |
| 4 | `llamada anidada f(g(x))` | `_t1 = CALL f, 1` |
| 5 | `CALL sin argumentos` | `_t0 = CALL p, 0` |
| 6 | `RETURN con valor` | `RETURN _t0` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 26 - Llamadas, parametros y retorno ===
Caso                               esperado:              obtenido:              resultado
[ 1] PARAM x                            esperado: PARAM x      obtenido: PARAM x      OK
[ 2] PARAM y                            esperado: PARAM y      obtenido: PARAM y      OK
[ 3] CALL con dos argumentos
     esperado: _t0 = CALL f, 2
     obtenido: _t0 = CALL f, 2
     -> OK
[ 4] llamada anidada f(g(x))
     esperado: _t1 = CALL f, 1
     obtenido: _t1 = CALL f, 1
     -> OK
[ 5] CALL sin argumentos
     esperado: _t0 = CALL p, 0
     obtenido: _t0 = CALL p, 0
     -> OK
[ 6] RETURN con valor                   esperado: RETURN _t0   obtenido: RETURN _t0   OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion26.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Los PARAM se emiten en el mismo orden en que se declararon los argumentos.
- [ ] El entero del CALL coincide con el número de PARAM realmente emitidos.
- [ ] Una llamada anidada conserva el temporal de la llamada interna y no lo sobrescribe.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion26_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion26.h
/* api_sesion26.h - contrato de la Sesion 26
 * Llamadas, parámetros y retorno
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION26_H
#define API_SESION26_H

int emitir_llamada(char lineas[][64], int max, int *n, const char *funcion, const char *a1, const char *a2, const char *resultado);
void emitir_return(char *dst, int cap, const char *valor);

#endif /* API_SESION26_H */


In [ ]:
%%writefile artesano_sesion26_c.c
/* artesano_sesion26_c.c - Fase 1: El Artesano (sin IA)
 * Llamadas, parámetros y retorno.
 *
 * Implementa lo que declara api_sesion26.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion26.h"
#include <stdio.h>
#include <string.h>

int emitir_llamada(char lineas[][64], int max, int *n, const char *funcion, const char *a1, const char *a2, const char *resultado) {
          (void)lineas; (void)max; (void)n; (void)funcion; (void)a1; (void)a2; (void)resultado; /* TODO: PARAM y CALL */
          return 0;
      }
      
      void emitir_return(char *dst, int cap, const char *valor) {
          (void)dst; (void)cap; (void)valor; /* TODO: RETURN con o sin valor */
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion26_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion26_c.c
/* ia_sesion26_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion26.h. No puede traer main().
 */
#include "api_sesion26.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion26_c.c
/* harness_sesion26_c.c - auditoria automatica de la Sesion 26
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion26_c.c artesano_sesion26_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion26.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 26 - Llamadas, parametros y retorno ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char l[16][64] = {{0}}; int n = 0; emitir_llamada(l, 16, &n, "f", "x", "y", "_t0"); verificar_str("PARAM x", "PARAM x", l[0]); }
    { char l[16][64] = {{0}}; int n = 0; emitir_llamada(l, 16, &n, "f", "x", "y", "_t0"); verificar_str("PARAM y", "PARAM y", l[1]); }
    { char l[16][64] = {{0}}; int n = 0; emitir_llamada(l, 16, &n, "f", "x", "y", "_t0"); verificar_str_largo("CALL con dos argumentos", "_t0 = CALL f, 2", l[2]); }
    { char l[16][64] = {{0}}; int n = 0; emitir_llamada(l, 16, &n, "g", "x", NULL, "_t0"); emitir_llamada(l, 16, &n, "f", "_t0", NULL, "_t1"); verificar_str_largo("llamada anidada f(g(x))", "_t1 = CALL f, 1", l[3]); }
    { char l[8][64] = {{0}}; int n = 0; emitir_llamada(l, 8, &n, "p", NULL, NULL, "_t0"); verificar_str_largo("CALL sin argumentos", "_t0 = CALL p, 0", l[0]); }
    { char b[32] = ""; emitir_return(b, 32, "_t0"); verificar_str("RETURN con valor", "RETURN _t0", b); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion26.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion26_c.c artesano_sesion26_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion26_c.c ia_sesion26_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion26.md` a `auditorias/auditoria_sesion26.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
